###  Q : 25 – Chargement et exploration des données

In [2]:

import pandas as pd

# 1. Charger le dataset
df_logements = pd.read_csv("TP3-logements .csv")

# 2. Afficher les premières lignes
display(df_logements.head())

# 3. Dimensions du dataset
print("Dimensions :", df_logements.shape)

# 4. Informations générales
df_logements.info()

# 5. Identifier les valeurs manquantes
print("\nValeurs manquantes :")
print(df_logements.isnull().sum())

# 6. Identifier les variables numériques
print("\nVariables numériques :")
print(df_logements.select_dtypes(include="number").columns.tolist())

# 7. Identifier les variables catégorielles
print("\nVariables catégorielles :")
print(df_logements.select_dtypes(include="object").columns.tolist())

# 8. Vérifier la cible
print("\nVariable cible : prix_dh")
print(df_logements["prix_dh"].describe())


,id_logement,surface_m2,anciennete_ans,distance_centre_km,quartier,prix_dh
0,L0001,105.6,17.0,8.2,Residentiel,1308954
1,L0002,110.6,9.0,19.5,Residentiel,1007062
2,L0003,134.3,5.0,1.7,Peripherie,1375932
3,L0004,74.1,19.0,19.5,Residentiel,800497
4,L0005,146.8,12.0,4.3,Residentiel,1726373


Dimensions : (800, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_logement         800 non-null    object 
 1   surface_m2          800 non-null    float64
 2   anciennete_ans      759 non-null    float64
 3   distance_centre_km  758 non-null    float64
 4   quartier            766 non-null    object 
 5   prix_dh             800 non-null    int64  
dtypes: float64(3), int64(1), object(2)
memory usage: 37.6+ KB

Valeurs manquantes :
id_logement            0
surface_m2             0
anciennete_ans        41
distance_centre_km    42
quartier              34
prix_dh                0
dtype: int64

Variables numériques :
['surface_m2', 'anciennete_ans', 'distance_centre_km', 'prix_dh']

Variables catégorielles :
['id_logement', 'quartier']

Variable cible : prix_dh
count    8.000000e+02
mean     1.224551e+06
std      4.768631e+

### Q 26 :  Séparation Train / Validation / Test

In [3]:

from sklearn.model_selection import train_test_split

# 1. Séparer les variables explicatives et la cible
X = df_logements.drop(columns=["prix_dh", "id_logement"])
y = df_logements["prix_dh"]

# 2. Séparation : 60% Train et 40% temporaire
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=42
)

# 3. Séparation des 40% : 20% Validation et 20% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

# 4. Vérifier les dimensions
print("Train :", X_train.shape)
print("Validation :", X_val.shape)
print("Test :", X_test.shape)


Train : (480, 4)
Validation : (160, 4)
Test : (160, 4)


### Q 27 :  Modèle de référence (DummyRegressor

In [4]:

from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Créer le modèle de référence
dummy = DummyRegressor(strategy="mean")

# Entraîner le modèle sur Train
dummy.fit(X_train, y_train)

# Prédictions sur Train et Validation
y_pred_dummy_train = dummy.predict(X_train)
y_pred_dummy_val = dummy.predict(X_val)

# Calculer MAE
mae_train = mean_absolute_error(y_train, y_pred_dummy_train)
mae_val = mean_absolute_error(y_val, y_pred_dummy_val)

# Afficher les résultats
print("Moyenne des prix Train :", y_train.mean())

print("\nMAE Train :", mae_train)
print("MAE Validation :", mae_val)


Moyenne des prix Train : 1251360.59375

MAE Train : 396404.00247395836
MAE Validation : 405418.059375


### Q28 : 28 : Pipeline + LinearRegression

In [5]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# 1. Identifier les variables
num_features = [
    "surface_m2",
    "anciennete_ans",
    "distance_centre_km"
]

cat_features = ["quartier"]

# 2. Pipeline pour les variables numériques
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 3. Pipeline pour les variables catégorielles
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# 4. Combiner les deux prétraitements
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

# 5. Créer le Pipeline complet
model_lr = Pipeline([
    ("preprocessing", preprocessor),
    ("regression", LinearRegression())
])

# 6. Entraîner le modèle
model_lr.fit(X_train, y_train)

# 7. Prédire les prix sur Validation
y_pred_lr = model_lr.predict(X_val)

# 8. Calculer MAE
mae_lr = mean_absolute_error(y_val, y_pred_lr)

print("MAE LinearRegression Validation :", mae_lr)


MAE LinearRegression Validation : 91847.24869060055


### Q29 :  Calcul des métriques de régression

In [6]:

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

# Calcul des métriques sur Validation
mae = mean_absolute_error(y_val, y_pred_lr)

mse = mean_squared_error(y_val, y_pred_lr)

rmse = np.sqrt(mse)

r2 = r2_score(y_val, y_pred_lr)

# Affichage
print("MAE :", mae)
print("MSE :", mse)
print("RMSE :", rmse)
print("R2 :", r2)


MAE : 91847.24869060055
MSE : 15199314943.213049
RMSE : 123285.50175593661
R2 : 0.931589969584252


### Q 30 : DecisionTreeRegressor

In [7]:

from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

# Tester trois profondeurs
depths = [2, 5, None]

for depth in depths:

    # Créer le modèle
    tree_model = Pipeline([
        ("preprocessing", preprocessor),
        ("regression", DecisionTreeRegressor(
            max_depth=depth,
            random_state=42
        ))
    ])

    # Entraîner sur Train
    tree_model.fit(X_train, y_train)

    # Prédictions Train et Validation
    pred_train = tree_model.predict(X_train)
    pred_val = tree_model.predict(X_val)

    # Calculer MAE
    mae_train = mean_absolute_error(y_train, pred_train)
    mae_val = mean_absolute_error(y_val, pred_val)

    print("max_depth =", depth)
    print("MAE Train :", mae_train)
    print("MAE Validation :", mae_val)
    print("---------------------------")


max_depth = 2
MAE Train : 190892.9634791463
MAE Validation : 213415.5156975162
---------------------------
max_depth = 5
MAE Train : 93308.53179000386
MAE Validation : 125615.25100841638
---------------------------
max_depth = None
MAE Train : 0.0
MAE Validation : 124131.35
---------------------------


### Q 31 : Analyse des prédictions de trois logements

In [8]:

# Prendre les 3 premiers logements de Validation
trois_logements = X_val.iloc[:3].copy()

# Créer un tableau de comparaison
resultats = pd.DataFrame({
    "Surface_m2": trois_logements["surface_m2"].values,
    "Quartier": trois_logements["quartier"].values,
    "Prix_reel": y_val.iloc[:3].values,
    "Prix_predit": y_pred_lr[:3]
})

# Calculer l'erreur absolue
resultats["Erreur_absolue"] = abs(
    resultats["Prix_reel"] - resultats["Prix_predit"]
)

# Afficher les résultats
display(resultats.round(2))


,Surface_m2,Quartier,Prix_reel,Prix_predit,Erreur_absolue
0,55.1,Peripherie,569055,445962.96,123092.04
1,172.2,NaN,1543048,1702591.28,159543.28
2,48.4,Peripherie,566456,471394.22,95061.78


### Q32 :  Sélection du meilleur modèle et évaluation finale

In [9]:

# 1. Combiner Train et Validation
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

print("Train + Validation :", X_train_val.shape)

# 2. Créer le modèle final
final_model = Pipeline([
    ("preprocessing", preprocessor),
    ("regression", LinearRegression())
])

# 3. Réentraîner sur Train + Validation
final_model.fit(X_train_val, y_train_val)

print("Modèle final entraîné avec succès !")

# 4. Évaluation finale sur Test
y_pred_test = final_model.predict(X_test)

mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, y_pred_test)

print("\n=== LinearRegression - Test ===")
print("MAE :", mae_test)
print("MSE :", mse_test)
print("RMSE :", rmse_test)
print("R2 :", r2_test)

# 5. Comparaison avec DummyRegressor
dummy_final = DummyRegressor(strategy="mean")
dummy_final.fit(X_train_val, y_train_val)

y_pred_dummy = dummy_final.predict(X_test)

print("\n=== DummyRegressor - Test ===")
print("MAE :", mean_absolute_error(y_test, y_pred_dummy))
print("MSE :", mean_squared_error(y_test, y_pred_dummy))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred_dummy)))
print("R2 :", r2_score(y_test, y_pred_dummy))


Train + Validation : (640, 4)
Modèle final entraîné avec succès !

=== LinearRegression - Test ===
MAE : 87660.73251657313
MSE : 14114204969.084087
RMSE : 118803.21952322709
R2 : 0.9380154398021487

=== DummyRegressor - Test ===
MAE : 392149.3006640625
MSE : 229700733649.6058
RMSE : 479271.0440341726
R2 : -0.008763793892851313
